# Speech JEPA 2×2: Prescribed Structure vs Free Space

{GMM, k-means} × {soft, hard} — frozen on log-mel. Runtime → **T4 GPU** → **Run All**.

All progress saves to Google Drive. If disconnected → just Run All again, it resumes.


In [2]:
!pip install -q datasets scikit-learn umap-learn

import os, math, random, pickle, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MiniBatchKMeans
from tqdm.auto import tqdm
from datasets import load_dataset

# ---- Seed ----
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ---- Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/speech_jepa_2x2'
os.makedirs(SAVE_DIR, exist_ok=True)

def save_to_drive(obj, filename):
    with open(os.path.join(SAVE_DIR, filename), 'wb') as f:
        pickle.dump(obj, f)
    print(f"  -> Saved: {filename}")

def load_from_drive(filename):
    path = os.path.join(SAVE_DIR, filename)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None

print(f"\nSave dir: {SAVE_DIR}")
for f in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"  {f} ({sz:.1f} MB)")

# ---- Constants ----
SAMPLE_RATE = 16000
N_MELS = 80
HOP_LENGTH = 320
N_FFT = 400
K = 512
DIM = 256
N_LAYERS = 4
N_HEADS = 4
N_STEPS = 2000
BATCH_SIZE = 24
N_TRAIN = 8000
CKPT_EVERY = 200

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT,
    hop_length=HOP_LENGTH, n_mels=N_MELS,
).to(DEVICE)

def extract_logmel(waveform, sr=SAMPLE_RATE):
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    waveform = waveform.unsqueeze(0).to(DEVICE)
    mel = mel_transform(waveform)
    return torch.log(mel + 1e-6).squeeze(0).cpu()

print("\nSetup complete.")


Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
Mounted at /content/drive

Save dir: /content/drive/MyDrive/speech_jepa_2x2
  calibration.pkl (0.0 MB)
  ckpt_Pure_JEPA.pkl (71.0 MB)
  clustering.pkl (1.6 MB)
  features.pkl (1680.3 MB)

Setup complete.


In [3]:
print("Loading LibriSpeech train-clean-100...")
ds = load_dataset("librispeech_asr", "clean", split="train.100")
print(f"Train: {len(ds)} utterances")

ds_dev = load_dataset("librispeech_asr", "clean", split="validation")
print(f"Dev: {len(ds_dev)} utterances")


Loading LibriSpeech train-clean-100...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

...

Generating validation split:   0%|          | 0/2703 [00:00<?, ? examples/s]

Train: 28539 utterances


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Dev: 2703 utterances


In [4]:
cached = load_from_drive('clustering.pkl')

if cached is not None and cached.get('kmeans') is not None:
    gmm = cached['gmm']
    kmeans = cached['kmeans']
    print(f"Loaded GMM + K-Means from Drive (K={gmm.n_components})")
else:
    # Need to fit — extract minimal features
    TARGET_FRAMES = 300_000
    print(f"Extracting features ({TARGET_FRAMES:,} frames)...")
    chunks = []
    total = 0
    for i in tqdm(range(min(2000, len(ds)))):
        audio = ds[i]['audio']
        waveform = torch.tensor(audio['array'], dtype=torch.float32)
        feat = extract_logmel(waveform, audio['sampling_rate']).T.numpy()
        chunks.append(feat)
        total += feat.shape[0]
        if total >= TARGET_FRAMES:
            break
    fit_data = np.concatenate(chunks, axis=0)[:TARGET_FRAMES].astype(np.float32)
    del chunks; gc.collect()
    print(f"fit_data: {fit_data.shape[0]:,} x {fit_data.shape[1]}, {fit_data.nbytes/1e6:.0f} MB")

    print(f"\nFitting GMM (K={K}, diag)...")
    gmm = GaussianMixture(
        n_components=K, covariance_type='diag',
        max_iter=50, n_init=1, init_params='k-means++',
        verbose=2, random_state=42, reg_covar=1e-4,
    )
    gmm.fit(fit_data)
    print(f"GMM converged: {gmm.converged_}")
    save_to_drive({'gmm': gmm, 'kmeans': None}, 'clustering.pkl')

    print(f"\nFitting K-Means (K={K})...")
    kmeans = MiniBatchKMeans(
        n_clusters=K, batch_size=4096,
        max_iter=100, random_state=42, verbose=1,
    )
    kmeans.fit(fit_data)
    save_to_drive({'gmm': gmm, 'kmeans': kmeans}, 'clustering.pkl')

    del fit_data; gc.collect()

print("Clustering ready.")


Loaded GMM + K-Means from Drive (K=128)
Clustering ready.


In [5]:
class FrozenGMMSoft:
    def __init__(self, gmm_model):
        self.K = gmm_model.n_components
        self.means = torch.tensor(gmm_model.means_, dtype=torch.float32)
        self.precisions = torch.tensor(gmm_model.precisions_, dtype=torch.float32)
        self.log_weights = torch.tensor(np.log(gmm_model.weights_ + 1e-10), dtype=torch.float32)
        self.log_det = torch.tensor(np.sum(np.log(gmm_model.precisions_), axis=1), dtype=torch.float32)
        self.D = gmm_model.means_.shape[1]
    def to(self, device):
        self.means = self.means.to(device)
        self.precisions = self.precisions.to(device)
        self.log_weights = self.log_weights.to(device)
        self.log_det = self.log_det.to(device)
        return self
    @torch.no_grad()
    def __call__(self, log_mel):
        if log_mel.dim() == 3:
            B, D, T = log_mel.shape
            x = log_mel.permute(0, 2, 1).reshape(-1, D)
        else:
            x = log_mel
        diff = x.unsqueeze(1) - self.means.unsqueeze(0)
        mahal = (diff ** 2 * self.precisions.unsqueeze(0)).sum(-1)
        log_prob = -0.5 * (self.D * math.log(2 * math.pi) - self.log_det.unsqueeze(0) + mahal)
        log_resp = log_prob + self.log_weights.unsqueeze(0)
        return F.softmax(log_resp, dim=-1)

class FrozenGMMHard:
    def __init__(self, gmm_soft):
        self.gmm_soft = gmm_soft; self.K = gmm_soft.K
    def to(self, device):
        self.gmm_soft.to(device); return self
    @torch.no_grad()
    def __call__(self, log_mel):
        soft = self.gmm_soft(log_mel)
        hard = torch.zeros_like(soft)
        hard.scatter_(1, soft.argmax(dim=-1, keepdim=True), 1.0)
        return hard

class FrozenKMeansSoft:
    def __init__(self, kmeans_model, tau=1.0):
        self.centroids = torch.tensor(kmeans_model.cluster_centers_, dtype=torch.float32)
        self.K = kmeans_model.n_clusters; self.tau = tau
    def to(self, device):
        self.centroids = self.centroids.to(device); return self
    @torch.no_grad()
    def __call__(self, log_mel):
        if log_mel.dim() == 3:
            B, D, T = log_mel.shape
            x = log_mel.permute(0, 2, 1).reshape(-1, D)
        else:
            x = log_mel
        dists = torch.cdist(x, self.centroids).pow(2)
        return F.softmax(-dists / self.tau, dim=-1)

class FrozenKMeansHard:
    def __init__(self, kmeans_soft):
        self.kmeans_soft = kmeans_soft; self.K = kmeans_soft.K
    def to(self, device):
        self.kmeans_soft.to(device); return self
    @torch.no_grad()
    def __call__(self, log_mel):
        if log_mel.dim() == 3:
            B, D, T = log_mel.shape
            x = log_mel.permute(0, 2, 1).reshape(-1, D)
        else:
            x = log_mel
        dists = torch.cdist(x, self.kmeans_soft.centroids).pow(2)
        hard = torch.zeros(x.shape[0], self.kmeans_soft.K, device=x.device)
        hard.scatter_(1, dists.argmin(dim=-1, keepdim=True), 1.0)
        return hard

# Calibrate tau
cached_cal = load_from_drive('calibration.pkl')
if cached_cal is not None:
    best_tau = cached_cal['best_tau']
    print(f"Loaded tau={best_tau} from Drive")
else:
    print("Calibrating k-means temperature...")
    sample = []
    for i in range(200):
        audio = ds[i]['audio']
        waveform = torch.tensor(audio['array'], dtype=torch.float32)
        feat = extract_logmel(waveform, audio['sampling_rate']).T
        sample.append(feat)
    sample_feats = torch.cat(sample, dim=0)[:30000].to(DEVICE)
    del sample; gc.collect()

    gmm_soft_gen = FrozenGMMSoft(gmm).to(DEVICE)
    gmm_post = gmm_soft_gen(sample_feats)
    gmm_entropy = -(gmm_post * torch.log(gmm_post + 1e-10)).sum(-1).mean().item()

    best_tau, best_diff = 1.0, float('inf')
    for tau in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]:
        km = FrozenKMeansSoft(kmeans, tau=tau).to(DEVICE)
        kp = km(sample_feats)
        ke = -(kp * torch.log(kp + 1e-10)).sum(-1).mean().item()
        diff = abs(ke - gmm_entropy)
        print(f"  tau={tau:6.1f} -> ent={ke:.3f} (diff={diff:.3f})")
        if diff < best_diff:
            best_diff = diff; best_tau = tau

    del sample_feats, gmm_post; torch.cuda.empty_cache(); gc.collect()
    save_to_drive({'best_tau': best_tau}, 'calibration.pkl')
    print(f"Selected tau={best_tau}")

print(f"\nTarget generators ready (tau={best_tau})")


Loaded tau=2.0 from Drive

Target generators ready (tau=2.0)


In [6]:
# ---- Encoder ----
class CNNFrontend(nn.Module):
    def __init__(self, n_mels=80, dim=256):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(n_mels, dim, 3, 1, 1), nn.GELU(), nn.Conv1d(dim, dim, 3, 1, 1), nn.GELU())
    def forward(self, x): return self.net(x)

class TransformerEnc(nn.Module):
    def __init__(self, dim=256, n_layers=4, n_heads=4, ff_mult=4, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, 2000, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, dim_feedforward=dim*ff_mult, dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = x + self.pos_embed[:, :x.shape[1], :]
        return self.norm(self.transformer(x)).permute(0, 2, 1)

class SpeechJEPA(nn.Module):
    def __init__(self, n_mels=80, dim=256, n_layers=4, n_heads=4, K=512):
        super().__init__()
        self.dim = dim; self.K = K
        self.frontend = CNNFrontend(n_mels, dim)
        self.encoder = TransformerEnc(dim, n_layers, n_heads)
        self.mask_token = nn.Parameter(torch.randn(dim) * 0.02)
        pred_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, dim_feedforward=dim*2, dropout=0.1, activation='gelu', batch_first=True)
        self.predictor = nn.TransformerEncoder(pred_layer, num_layers=1)
        self.pred_proj = nn.Linear(dim, dim)
        self.cluster_head = nn.Sequential(nn.Linear(dim, dim), nn.GELU(), nn.LayerNorm(dim), nn.Linear(dim, K))
    def forward(self, log_mel, mask=None):
        z = self.frontend(log_mel)
        z_online = self.encoder(z)
        cluster_logits = self.cluster_head(z_online.permute(0, 2, 1))
        if mask is not None:
            z_masked = z_online.clone()
            me = mask.unsqueeze(1).expand_as(z_masked)
            z_masked = z_masked * (1 - me.float()) + self.mask_token.view(1,-1,1) * me.float()
        else:
            z_masked = z_online
        z_pred = self.predictor(z_masked.permute(0, 2, 1))
        z_pred = self.pred_proj(z_pred).permute(0, 2, 1)
        return z_online, z_pred, cluster_logits

class TargetEncoderEMA(nn.Module):
    def __init__(self, n_mels=80, dim=256, n_layers=4, n_heads=4):
        super().__init__()
        self.frontend = CNNFrontend(n_mels, dim)
        self.encoder = TransformerEnc(dim, n_layers, n_heads)
    def forward(self, log_mel):
        return self.encoder(self.frontend(log_mel))

@torch.no_grad()
def ema_update(target, online, decay=0.996):
    tp = dict(target.named_parameters())
    op = dict(online.named_parameters())
    for name in tp:
        if name in op:
            tp[name].data.mul_(decay).add_(op[name].data, alpha=1-decay)

# ---- Dataset ----
class SpeechDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        audio = self.ds[idx]['audio']
        waveform = torch.tensor(audio['array'], dtype=torch.float32)
        sr = audio['sampling_rate']
        if sr != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
        return waveform

def collate_fn(batch):
    max_len = max(w.shape[0] for w in batch)
    padded = torch.zeros(len(batch), max_len)
    for i, w in enumerate(batch):
        padded[i, :w.shape[0]] = w
    return padded

def create_block_mask(B, T, device):
    mask = torch.zeros(B, T, device=device)
    for b in range(B):
        ratio = random.uniform(0.4, 0.65)
        target = int(T * ratio); count = 0; att = 0
        while count < target and att < 100:
            span = min(random.randint(10, 25), T)
            start = random.randint(0, max(0, T - span))
            mask[b, start:start+span] = 1
            count = mask[b].sum().int().item(); att += 1
    return mask

# ---- Training function ----
def train_jepa(cond_name, tgt_gen, ds_train):
    set_seed(42)
    safe = cond_name.replace(' ', '_').replace('+', '_')

    online = SpeechJEPA(N_MELS, DIM, N_LAYERS, N_HEADS, K).to(DEVICE)
    target_enc = TargetEncoderEMA(N_MELS, DIM, N_LAYERS, N_HEADS).to(DEVICE)
    ema_update(target_enc, online, decay=0.0)

    opt = torch.optim.AdamW(online.parameters(), lr=3e-4, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_STEPS, eta_min=3e-6)
    hist = {'jepa_loss': [], 'cluster_loss': [], 'total_loss': [], 'lambda': []}
    start = 0

    ckpt = load_from_drive(f'ckpt_{safe}.pkl')
    if ckpt is not None:
        online.load_state_dict(ckpt['online_state'])
        target_enc.load_state_dict(ckpt['target_state'])
        opt.load_state_dict(ckpt['optimizer_state'])
        sched.load_state_dict(ckpt['scheduler_state'])
        hist = ckpt['history']
        start = ckpt['step'] + 1
        if start >= N_STEPS:
            print(f"  {cond_name}: already done ({start}>={N_STEPS}). Skipping.")
            return online, target_enc, hist

    print(f"\n{'='*50}\n{cond_name} [step {start} -> {N_STEPS}]\n{'='*50}")

    dl = torch.utils.data.DataLoader(
        SpeechDataset(ds_train), batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True, drop_last=True)
    dl_iter = iter(dl)
    pbar = tqdm(range(start, N_STEPS), desc=cond_name)

    for step in pbar:
        try:
            wavs = next(dl_iter)
        except StopIteration:
            dl_iter = iter(dl); wavs = next(dl_iter)

        wavs = wavs.to(DEVICE)
        with torch.no_grad():
            log_mel = torch.log(mel_transform(wavs) + 1e-6)
        B, _, T = log_mel.shape
        if T < 30: continue

        mask = create_block_mask(B, T, DEVICE)
        with torch.no_grad():
            z_tgt = target_enc(log_mel)

        z_on, z_pr, c_log = online(log_mel, mask)
        m3 = mask.unsqueeze(1).expand_as(z_pr)
        jepa_loss = ((z_pr - z_tgt.detach())**2 * m3).sum() / m3.sum().clamp_min(1)

        if tgt_gen is not None:
            # Get cluster targets per-frame from frozen generator
            # log_mel is (B, 80, T). We need per-frame targets aligned with c_log (B, T, K)
            T_enc = c_log.shape[1]
            with torch.no_grad():
                lm_flat = log_mel[:, :, :T_enc].permute(0, 2, 1).reshape(-1, N_MELS)
                q_flat = tgt_gen(lm_flat)  # (B*T_enc, K)
            mf = mask[:, :T_enc].reshape(-1).bool()
            plp = F.log_softmax(c_log[:, :T_enc, :].reshape(-1, K), dim=-1)
            if mf.any():
                pm, tm = plp[mf], q_flat[mf]
                if tm.max() == 1.0 and (tm.sum(-1) - 1.0).abs().max() < 1e-5:
                    cl_loss = F.nll_loss(pm, tm.argmax(-1))
                else:
                    cl_loss = F.kl_div(pm, tm, reduction='batchmean')
            else:
                cl_loss = torch.tensor(0.0, device=DEVICE)
            lam = 1.0 + (0.01 - 1.0) * (step / N_STEPS)
            total = jepa_loss + lam * cl_loss
        else:
            cl_loss = torch.tensor(0.0, device=DEVICE); lam = 0.0
            total = jepa_loss

        opt.zero_grad(); total.backward()
        torch.nn.utils.clip_grad_norm_(online.parameters(), 1.0)
        opt.step(); sched.step()
        ema_update(target_enc, online)

        hist['jepa_loss'].append(jepa_loss.item())
        hist['cluster_loss'].append(cl_loss.item())
        hist['total_loss'].append(total.item())
        hist['lambda'].append(lam)

        if step % 100 == 0:
            pbar.set_postfix(j=f"{jepa_loss.item():.4f}", c=f"{cl_loss.item():.4f}")
            torch.cuda.empty_cache()

        if (step+1) % CKPT_EVERY == 0 or step == N_STEPS - 1:
            save_to_drive({
                'step': step, 'online_state': online.state_dict(),
                'target_state': target_enc.state_dict(),
                'optimizer_state': opt.state_dict(),
                'scheduler_state': sched.state_dict(),
                'history': hist,
            }, f'ckpt_{safe}.pkl')

    return online, target_enc, hist

print("Encoder + training function ready.")
_m = SpeechJEPA(N_MELS, DIM, N_LAYERS, N_HEADS, K).to(DEVICE)
print(f"Params: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M")
del _m; torch.cuda.empty_cache()


Encoder + training function ready.
Params: 4.7M


In [7]:
gmm_soft = FrozenGMMSoft(gmm).to(DEVICE)
gmm_hard = FrozenGMMHard(gmm_soft).to(DEVICE)
km_soft = FrozenKMeansSoft(kmeans, tau=best_tau).to(DEVICE)
km_hard = FrozenKMeansHard(km_soft).to(DEVICE)

CONDITIONS = {
    'Pure JEPA': None,
    'GMM+Soft': gmm_soft,
    'GMM+Hard': gmm_hard,
    'KM+Soft': km_soft,
    'KM+Hard': km_hard,
}

ds_sub = ds.select(range(min(N_TRAIN, len(ds))))
print(f"Training {len(CONDITIONS)} conditions, {N_STEPS} steps, {len(ds_sub)} utterances, ckpt every {CKPT_EVERY}")

results = {}
trained_models = {}
for name, tg in CONDITIONS.items():
    model, tgt_enc, hist = train_jepa(name, tg, ds_sub)
    trained_models[name] = (model, tgt_enc)
    results[name] = hist
    torch.cuda.empty_cache(); gc.collect()

save_to_drive(results, 'training_histories.pkl')
print("\n" + "="*50 + "\nALL CONDITIONS COMPLETE\n" + "="*50)


Training 5 conditions, 2000 steps, 8000 utterances, ckpt every 200
  Pure JEPA: already done (2000>=2000). Skipping.

GMM+Soft [step 0 -> 2000]


GMM+Soft:   0%|          | 0/2000 [00:00<?, ?it/s]

  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl
  -> Saved: ckpt_GMM_Soft.pkl

GMM+Hard [step 0 -> 2000]


GMM+Hard:   0%|          | 0/2000 [00:00<?, ?it/s]

  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl
  -> Saved: ckpt_GMM_Hard.pkl

KM+Soft [step 0 -> 2000]


KM+Soft:   0%|          | 0/2000 [00:00<?, ?it/s]

  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl
  -> Saved: ckpt_KM_Soft.pkl

KM+Hard [step 0 -> 2000]


KM+Hard:   0%|          | 0/2000 [00:00<?, ?it/s]

  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: ckpt_KM_Hard.pkl
  -> Saved: training_histories.pkl

ALL CONDITIONS COMPLETE


In [8]:
def evaluate_clustering(model, ds_eval, n_utt=500):
    model.eval()
    preds_all = []; embs_all = []
    with torch.no_grad():
        for i in tqdm(range(min(n_utt, len(ds_eval))), desc="Eval", leave=False):
            audio = ds_eval[i]['audio']
            waveform = torch.tensor(audio['array'], dtype=torch.float32)
            sr = audio['sampling_rate']
            if sr != SAMPLE_RATE:
                waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
            waveform = waveform.unsqueeze(0).to(DEVICE)
            lm = torch.log(mel_transform(waveform) + 1e-6)
            if lm.shape[-1] < 10: continue
            zo, _, cl = model(lm)
            preds_all.append(cl.squeeze(0).argmax(-1).cpu())
            embs_all.append(zo.squeeze(0).permute(1,0).cpu())
    ap = torch.cat(preds_all); ae = torch.cat(embs_all)
    counts = torch.bincount(ap, minlength=K).float()
    probs = counts / counts.sum()
    pnz = probs[probs > 0]
    nent = (-(pnz * torch.log(pnz)).sum().item()) / math.log(K) * 100
    active = (counts > 0).sum().item()
    cons = [((p[1:]==p[:-1]).float().mean().item()) for p in preds_all if len(p)>1]
    return {'entropy_pct': nent, 'active_clusters': active,
            'adj_consistency': np.mean(cons), 'counts': counts.numpy(),
            'embeddings': ae, 'cluster_preds': ap}

print("Evaluating...")
eval_results = {}
for name, (model, _) in trained_models.items():
    eval_results[name] = evaluate_clustering(model, ds_dev)
    r = eval_results[name]
    print(f"  {name}: entropy={r['entropy_pct']:.1f}%, active={r['active_clusters']}/{K}")

print("\n" + "="*60)
print(f"{'Condition':<15} {'Entropy%':>10} {'Active':>8}")
print("-"*60)
for n in CONDITIONS:
    r = eval_results[n]
    print(f"{n:<15} {r['entropy_pct']:>9.1f}% {r['active_clusters']:>7}/{K}")
print("="*60)

eval_save = {k: {kk: vv for kk, vv in v.items() if kk not in ('embeddings','cluster_preds')}
             for k, v in eval_results.items()}
save_to_drive(eval_save, 'eval_results.pkl')


Evaluating...


Eval:   0%|          | 0/500 [00:00<?, ?it/s]

  Pure JEPA: entropy=53.2%, active=138/512


Eval:   0%|          | 0/500 [00:00<?, ?it/s]

  GMM+Soft: entropy=71.5%, active=128/512


...

  GMM+Hard: entropy=71.5%, active=128/512


Eval:   0%|          | 0/500 [00:00<?, ?it/s]

  KM+Soft: entropy=73.7%, active=128/512


Eval:   0%|          | 0/500 [00:00<?, ?it/s]

  KM+Hard: entropy=73.7%, active=128/512

Condition         Entropy%   Active
------------------------------------------------------------
Pure JEPA            53.2%     138/512
GMM+Soft             71.5%     128/512
GMM+Hard             71.5%     128/512
KM+Soft              73.7%     128/512
KM+Hard              73.7%     128/512
  -> Saved: eval_results.pkl


In [9]:
names = list(CONDITIONS.keys())
colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

# Fig 1: Metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ents = [eval_results[n]['entropy_pct'] for n in names]
bars = axes[0].bar(range(len(names)), ents, color=colors, alpha=0.8)
axes[0].set_ylabel('Entropy (%)'); axes[0].set_title('Cluster Entropy')
axes[0].set_xticks(range(len(names))); axes[0].set_xticklabels(names, rotation=45, ha='right', fontsize=8)
for b, v in zip(bars, ents):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{v:.1f}%', ha='center', fontsize=9)
act = [eval_results[n]['active_clusters'] for n in names]
axes[1].bar(range(len(names)), act, color=colors, alpha=0.8)
axes[1].set_ylabel(f'Active/{K}'); axes[1].set_title('Active Clusters')
axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels(names, rotation=45, ha='right', fontsize=8)
for i, n in enumerate(names):
    c = np.sort(eval_results[n]['counts'])[::-1]
    axes[2].semilogy(c+1, color=colors[i], alpha=0.8, label=n)
axes[2].legend(fontsize=7); axes[2].set_title('Distribution')
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'cluster_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

# Fig 2: Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, n in enumerate(names):
    j = np.convolve(results[n]['jepa_loss'], np.ones(50)/50, mode='valid')
    axes[0].plot(j, color=colors[i], alpha=0.8, label=n)
    if max(results[n]['cluster_loss']) > 0:
        c = np.convolve(results[n]['cluster_loss'], np.ones(50)/50, mode='valid')
        axes[1].plot(c, color=colors[i], alpha=0.8, label=n)
axes[0].set_title('JEPA Loss'); axes[0].legend(fontsize=7); axes[0].set_yscale('log')
axes[1].set_title('Cluster Loss'); axes[1].legend(fontsize=7)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# Fig 3: Heatmap
e = {n: eval_results[n]['entropy_pct'] for n in names}
data = np.array([[e['GMM+Soft'], e['GMM+Hard']], [e['KM+Soft'], e['KM+Hard']]])
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(data, cmap='YlOrRd', aspect='auto')
ax.set_xticks([0,1]); ax.set_xticklabels(['Soft','Hard'])
ax.set_yticks([0,1]); ax.set_yticklabels(['GMM','K-Means'])
ax.set_title('2x2: Cluster Entropy (%)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{data[i,j]:.1f}%', ha='center', va='center', fontsize=14, fontweight='bold')
plt.colorbar(im); plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'factorial_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Plots saved to Drive.")


Plots saved to Drive.


In [10]:
e = {n: eval_results[n]['entropy_pct'] for n in names}
sm = (e['GMM+Soft']+e['KM+Soft'])/2
hm = (e['GMM+Hard']+e['KM+Hard'])/2
gm = (e['GMM+Soft']+e['GMM+Hard'])/2
km = (e['KM+Soft']+e['KM+Hard'])/2
am = (e['GMM+Soft']+e['GMM+Hard']+e['KM+Soft']+e['KM+Hard'])/4

print("2x2 Factor Analysis")
print("="*50)
print(f"  Soft vs Hard:     {sm:.1f}% vs {hm:.1f}% (D={sm-hm:+.1f}%)")
print(f"  GMM vs KM:        {gm:.1f}% vs {km:.1f}% (D={gm-km:+.1f}%)")
print(f"  Anchored vs Pure: {am:.1f}% vs {e['Pure JEPA']:.1f}% (D={am-e['Pure JEPA']:+.1f}%)")
if abs(sm-hm) > abs(gm-km):
    print("  -> Assignment type matters MORE")
else:
    print("  -> Clustering method matters MORE")
if am - e['Pure JEPA'] > 20:
    print("  -> Prescribed structure is the dominant factor")


2x2 Factor Analysis
  Soft vs Hard:     72.6% vs 72.6% (D=+0.0%)
  GMM vs KM:        71.5% vs 73.7% (D=-2.2%)
  Anchored vs Pure: 72.6% vs 53.2% (D=+19.4%)
  -> Clustering method matters MORE


In [11]:
save_to_drive({
    'eval_results': {k: {kk: vv for kk, vv in v.items() if kk not in ('embeddings','cluster_preds')}
                     for k, v in eval_results.items()},
    'training_history': results,
    'config': {'K': K, 'N_STEPS': N_STEPS, 'DIM': DIM, 'best_tau': best_tau},
}, 'final_results.pkl')

# --- Download to your laptop ---
from google.colab import files
import zipfile

zip_path = '/content/speech_jepa_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(SAVE_DIR):
        fpath = os.path.join(SAVE_DIR, fname)
        if not os.path.isfile(fpath): continue
        size = os.path.getsize(fpath) / 1e6
        if fname.endswith('.png') or fname in ('eval_results.pkl','final_results.pkl','calibration.pkl'):
            zf.write(fpath, fname)
            print(f"  Added: {fname} ({size:.1f} MB)")
        else:
            print(f"  Skipped: {fname} ({size:.1f} MB)")

print(f"\nZip: {os.path.getsize(zip_path)/1e6:.1f} MB")
print("Downloading...")
files.download(zip_path)


  -> Saved: final_results.pkl
  Skipped: features.pkl (1680.3 MB)
  Skipped: clustering.pkl (1.6 MB)
  Added: calibration.pkl (0.0 MB)
  Skipped: ckpt_Pure_JEPA.pkl (71.0 MB)
  Skipped: ckpt_GMM_Soft.pkl (72.6 MB)
  Skipped: ckpt_GMM_Hard.pkl (72.6 MB)
  Skipped: ckpt_KM_Soft.pkl (72.6 MB)
  Skipped: ckpt_KM_Hard.pkl (72.6 MB)
  Skipped: training_histories.pkl (0.4 MB)
  Added: eval_results.pkl (0.0 MB)
  Added: cluster_metrics.png (0.1 MB)
  Added: training_curves.png (0.1 MB)
  Added: factorial_heatmap.png (0.0 MB)
  Added: final_results.pkl (0.4 MB)

Zip: 0.4 MB
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>